In [1]:
from tqdm import tqdm
import numpy as np
import warnings
from scipy.stats import ConstantInputWarning

from analyses.response_window_analysis import run_permutation_anova_by_window
from analyses.spike_count import extract_spike_counts_from_windows

warnings.simplefilter("ignore", ConstantInputWarning)
from analyses.response_window_finder.threshold_window_detection import compute_timebinned_spikecount_per_neuron, z_score, \
    threshold_and_fill_gap, extract_consecutive_ranges, remove_consecutive_tuples, \
    find_corresponding_values_for_index_ranges
%load_ext autoreload
%autoreload 2

In [2]:
date = "2023-09-26"
round_no = 1
bin_size = 0.05  # in sec
rounded_time = np.round(np.arange(bin_size, 3.50, bin_size), 2)
monkey_group = 'Zombies'
results = []
zombies_timebin_spikecount_list= compute_timebinned_spikecount_per_neuron(date, round_no, bin_size, 'Zombies')
for _, r in tqdm(zombies_timebin_spikecount_list.iterrows(), total=len(zombies_timebin_spikecount_list), desc="Processing each neuron"):
    data = r['TotalSpikeCountList']
    neuron = r['NeuronID']
    normalized_data = z_score(data)
    thresh = 0.5
    change_points = threshold_and_fill_gap(normalized_data, thresh)
    windows = extract_consecutive_ranges(change_points)
    filtered_windows = remove_consecutive_tuples(windows)
    time_windows = find_corresponding_values_for_index_ranges(filtered_windows, rounded_time)
    if len(time_windows) > 0:
        print(f"---------------- {neuron} ----------------")
        print(time_windows)
        for start_time, end_time in time_windows:
            results.append({
                'NeuronID': neuron,
                'WindowStart_ms': int(start_time * 1000),
                'WindowEnd_ms': int(end_time * 1000)
            })



Reading Intan Technologies RHD2000 Data File, Version 3.2

Found 24 amplifier channels.
Found 0 auxiliary input channels.
Found 0 supply voltage channels.
Found 0 board ADC channels.
Found 2 board digital input channels.
Found 0 board digital output channels.
Found 0 temperature sensors channels.

Header file contains no data.  Amplifiers were sampled at 20.00 kS/s.
Done!  Elapsed time: 0.0 seconds


Processing each neuron: 100%|██████████| 32/32 [00:00<00:00, 9504.83it/s]

---------------- 2023-09-26_1_Channel.C_003 ----------------
[(0.65, 0.8)]
---------------- 2023-09-26_1_Channel.C_004 ----------------
[(0.05, 0.35), (0.65, 1.0)]
---------------- 2023-09-26_1_Channel.C_005_Unit 1 ----------------
[(0.3, 0.4), (0.7, 0.9)]
---------------- 2023-09-26_1_Channel.C_006 ----------------
[(0.65, 0.85)]
---------------- 2023-09-26_1_Channel.C_010 ----------------
[(0.65, 0.8)]
---------------- 2023-09-26_1_Channel.C_011_Unit 1 ----------------
[(0.35, 0.5)]
---------------- 2023-09-26_1_Channel.C_012_Unit 3 ----------------
[(0.35, 0.65), (0.75, 0.85), (1.7, 1.9)]
---------------- 2023-09-26_1_Channel.C_014_Unit 1 ----------------
[(0.15, 0.5)]
---------------- 2023-09-26_1_Channel.C_017_Unit 1 ----------------
[(0.15, 0.25), (0.35, 0.5)]
---------------- 2023-09-26_1_Channel.C_018_Unit 1 ----------------
[(0.05, 0.15), (1.75, 1.85)]
---------------- 2023-09-26_1_Channel.C_018_Unit 2 ----------------
[(0.1, 0.2), (0.35, 0.45), (0.7, 0.8)]
---------------- 20

In [3]:
import pandas as pd
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=['NeuronID'])
results_df.head()

,NeuronID,WindowStart_ms,WindowEnd_ms
0,2023-09-26_1_Channel.C_003,650,800
1,2023-09-26_1_Channel.C_004,50,350
2,2023-09-26_1_Channel.C_004,650,1000
3,2023-09-26_1_Channel.C_005_Unit 1,300,400
4,2023-09-26_1_Channel.C_005_Unit 1,700,900


In [4]:
final_df = extract_spike_counts_from_windows(results_df)

Extracting spike counts:   0%|          | 0/35 [00:00<?, ?it/s]

2023-09-26_1_Channel.C_003
('2023-09-26', 1)
cache_key not in cache. reading the raw file

Reading Intan Technologies RHD2000 Data File, Version 3.2

Found 24 amplifier channels.
Found 0 auxiliary input channels.
Found 0 supply voltage channels.
Found 0 board ADC channels.
Found 2 board digital input channels.
Found 0 board digital output channels.
Found 0 temperature sensors channels.

Header file contains no data.  Amplifiers were sampled at 20.00 kS/s.
Done!  Elapsed time: 0.0 seconds


Extracting spike counts:  60%|██████    | 21/35 [00:04<00:01,  7.02it/s]

2023-09-26_1_Channel.C_004
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_004
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_005_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_005_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_006
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_010
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_011_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_012_Unit 3
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_012_Unit 3
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_012_Unit 3
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_014_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_017_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_017_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_018_Unit 1
('2023-09-26', 1)
c

Extracting spike counts: 100%|██████████| 35/35 [00:05<00:00,  6.89it/s]

2023-09-26_1_Channel.C_020_Unit 2
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_021
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_022_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_022_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_024
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_025_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_025_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_025_Unit 2
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_027_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_027_Unit 1
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_028
('2023-09-26', 1)
cache_key in cache! :)
2023-09-26_1_Channel.C_029
('2023-09-26', 1)
cache_key in cache! :)


In [5]:
final_df

,NeuronID,MonkeyName,MonkeyGroup,TaskField,WindowStart_ms,WindowEnd_ms,SpikeCount
0,2023-09-26_1_Channel.C_003,114J,Instigators,1695747327712000,650,800,0
1,2023-09-26_1_Channel.C_003,69X,Zombies,1695747327797000,650,800,0
2,2023-09-26_1_Channel.C_003,59E,Instigators,1695747327892000,650,800,0
3,2023-09-26_1_Channel.C_003,40J,Stranger Things,1695747327973000,650,800,0
4,2023-09-26_1_Channel.C_003,40J,Stranger Things,1695747328008000,650,800,0
...,...,...,...,...,...,...,...
12245,2023-09-26_1_Channel.C_029,19J,Best Frans,1695747353785000,650,800,0
12246,2023-09-26_1_Channel.C_029,101G,Best Frans,1695747353875000,650,800,0
12247,2023-09-26_1_Channel.C_029,G942,Instigators,1695747353948000,650,800,0
12248,2023-09-26_1_Channel.C_029,48Z,Instigators,1695747354017000,650,800,0


In [6]:
zombies_df = final_df[final_df['MonkeyGroup'] == 'Zombies']
run_permutation_anova_by_window(zombies_df,
                                    category_col='MonkeyName',
                                    neuron_col='NeuronID',
                                    count_col='SpikeCount',
                                    window_start_col='WindowStart_ms',
                                    window_end_col='WindowEnd_ms',
                                    n_permutations=1000,
                                    alpha=0.05,
                                    plot=True)

Running Perm ANOVA per (Neuron, Window): 100%|██████████| 35/35 [00:04<00:00,  8.13it/s]


All Results:
                             NeuronID  WindowStart_ms  WindowEnd_ms  \
0          2023-09-26_1_Channel.C_003             650           800   
1          2023-09-26_1_Channel.C_004              50           350   
2          2023-09-26_1_Channel.C_004             650          1000   
3   2023-09-26_1_Channel.C_005_Unit 1             300           400   
4   2023-09-26_1_Channel.C_005_Unit 1             700           900   
5          2023-09-26_1_Channel.C_006             650           850   
6          2023-09-26_1_Channel.C_010             650           800   
7   2023-09-26_1_Channel.C_011_Unit 1             350           500   
8   2023-09-26_1_Channel.C_012_Unit 3             350           650   
9   2023-09-26_1_Channel.C_012_Unit 3             750           850   
10  2023-09-26_1_Channel.C_012_Unit 3            1700          1900   
11  2023-09-26_1_Channel.C_014_Unit 1             150           500   
12  2023-09-26_1_Channel.C_017_Unit 1             150          

,NeuronID,WindowStart_ms,WindowEnd_ms,F-statistic,p-value
0,2023-09-26_1_Channel.C_003,650,800,1.303195,0.141
1,2023-09-26_1_Channel.C_004,50,350,1.741725,0.105
2,2023-09-26_1_Channel.C_004,650,1000,1.550956,0.156
3,2023-09-26_1_Channel.C_005_Unit 1,300,400,0.815823,0.479
4,2023-09-26_1_Channel.C_005_Unit 1,700,900,1.006339,0.439
5,2023-09-26_1_Channel.C_006,650,850,1.413942,0.025
6,2023-09-26_1_Channel.C_010,650,800,1.532373,0.033
7,2023-09-26_1_Channel.C_011_Unit 1,350,500,1.029580,0.414
8,2023-09-26_1_Channel.C_012_Unit 3,350,650,0.782943,0.650
9,2023-09-26_1_Channel.C_012_Unit 3,750,850,0.796947,0.557
